# 실습 1-1. OpenAI Streaming 호출 직접 다뤄보기

## 실습 목표

이번 실습은 FastAPI에 들어가기 전에 OpenAI Streaming 호출을 **손으로 직접 다뤄보는** 단계입니다. `stream=True` 옵션 하나로 어떻게 응답이 달라지는지, chunk를 어떻게 받고 누적하는지, 토큰 사용량은 어떻게 챙기는지를 확인합니다.

구체적으로 다음을 차례로 확인합니다.

- `stream=True`로 OpenAI Streaming 호출
- chunk 흐름 한 줄씩 출력해서 구조 확인
- `delta.content`만 뽑아 실시간 출력
- chunk 누적해서 완성본 만들기
- `stream_options`로 토큰 사용량 받기

## 1. 클라이언트 준비

LLM 클라이언트는 교육 환경에서 제공되는 MLAPI 접속 정보를 사용합니다. 3일차에서 쓰던 패턴 그대로, `.env`에서 먼저 읽고 없으면 입력 프롬프트로 받습니다.

In [4]:
import os
from getpass import getpass
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

base_url   = os.getenv("MLAPI_BASE_URL", "https://mlapi.run/8ebfcef5-8d34-4355-a8d9-89f9253acd2b/v1")
api_key    = os.getenv("MLAPI_API_KEY")
model_name = os.getenv("MLAPI_MODEL", "openai/gpt-5.4")

if not api_key or api_key.startswith("여기에"):
    api_key = getpass("MLAPI API Key를 입력하세요: ")

client = OpenAI(base_url=base_url, api_key=api_key)

print("Base URL:", client.base_url)
print("모델명:", model_name)

Base URL: https://mlapi.run/8ebfcef5-8d34-4355-a8d9-89f9253acd2b/v1/
모델명: openai/gpt-5.4


## 2. `stream=True`로 호출하고 chunk 한 줄씩 출력

`chat.completions.create(...)`에 `stream=True`를 추가하면 반환 값이 **chunk를 흘려보내는 iterator**로 바뀝니다. 일단 chunk 자체를 그대로 출력해 구조를 확인해봅시다.

In [5]:
response = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": "스트리밍 응답을 한 줄로 설명해줘."}],
    stream=True,
)

for i, chunk in enumerate(response):
    print(f"[{i}]", chunk)
    if i >= 5:
        break

[0] ChatCompletionChunk(id='chatcmpl-Dpw6jMfbeebEZChrJhAQhuDanjc5p', choices=[Choice(delta=ChoiceDelta(content='', function_call=None, refusal=None, role='assistant', tool_calls=None), finish_reason=None, index=0, logprobs=None)], created=1781270465, model='gpt-5.4-2026-03-05', object='chat.completion.chunk', moderation=None, service_tier='default', system_fingerprint=None, usage=None, obfuscation='35vWR')
[1] ChatCompletionChunk(id='chatcmpl-Dpw6jMfbeebEZChrJhAQhuDanjc5p', choices=[Choice(delta=ChoiceDelta(content='스트', function_call=None, refusal=None, role=None, tool_calls=None), finish_reason=None, index=0, logprobs=None)], created=1781270465, model='gpt-5.4-2026-03-05', object='chat.completion.chunk', moderation=None, service_tier='default', system_fingerprint=None, usage=None, obfuscation='1Ebit')
[2] ChatCompletionChunk(id='chatcmpl-Dpw6jMfbeebEZChrJhAQhuDanjc5p', choices=[Choice(delta=ChoiceDelta(content='리', function_call=None, refusal=None, role=None, tool_calls=None), finish

포인트

- `response`는 객체가 아니라 iterator. `for chunk in response:` 패턴으로 하나씩 꺼냄
- 첫 chunk는 보통 `delta.role="assistant"`만, 그 다음부터 `delta.content`에 텍스트가 담겨 옴
- 환경에 따라 마지막 chunk는 `choices`가 비어 있는 채로 `usage`만 담겨 오기도 함 → 다음 셀에서 가드 필요

## 3. `delta.content`만 뽑아 실시간 출력

전체 chunk 객체가 아니라 **이번에 생성된 텍스트 조각**만 뽑아 화면에 흘려보냅니다. `print(..., end="", flush=True)`가 핵심 — 줄바꿈 없이, 버퍼링 없이 즉시 출력.

한 가지 주의할 점은 **마지막 chunk에서 `choices` 리스트가 비어 있을 수 있다**는 것입니다. 백엔드 종류에 따라 마지막에 usage만 담긴 빈 chunk가 도착할 수 있으므로 `if not chunk.choices: continue` 가드를 넣어둡니다.

In [6]:
response = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": "스트리밍 응답을 짧게 설명해줘."}],
    stream=True,
)

for chunk in response:
    if not chunk.choices:
        continue
    piece = chunk.choices[0].delta.content
    if piece:
        print(piece, end="", flush=True)
print()

스트리밍 응답은 서버가 **응답 전체를 한 번에 보내지 않고**, 준비되는 내용부터 **조금씩 나눠서 보내는 방식**입니다.

예를 들면:

- 일반 응답: 답이 완성된 뒤 한 번에 전달
- 스트리밍 응답: 답이 생성되는 동안 문장이나 토큰 단위로 바로 전달

장점:
- **첫 응답이 빠르게 보임**
- 긴 답변도 **기다리는 느낌이 줄어듦**
- 실시간 출력처럼 보여서 **사용자 경험이 좋아짐**

예:
채팅 AI가  
“안녕하세요. 이 문제는…”  
처럼 글자를 점점 이어서 보여주는 것이 스트리밍 응답입니다.


포인트

- `if not chunk.choices: continue` — `choices`가 빈 리스트인 chunk를 안전하게 건너뜀
- `delta.content`는 종종 `None`으로 옴 (첫 chunk, 중간 빈 chunk 등) → `if piece:` 한 줄로 처리
- 글자가 흐르듯 출력되는 게 보이면 스트리밍이 제대로 동작하는 것

## 4. chunk 누적해서 완성본 만들기

받은 chunk를 그대로 흘려보내기만 하면 백엔드에서 로그·DB 저장 같은 후처리를 못 합니다. **누적 변수**에 담아 완성본을 만들어두는 패턴이 표준입니다.

In [7]:
response = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": "FastAPI를 한 줄로 설명해줘."}],
    stream=True,
)

collected = []
for chunk in response:
    if not chunk.choices:
        continue
    piece = chunk.choices[0].delta.content
    if piece:
        collected.append(piece)
        print(piece, end="", flush=True)
print()

full_text = "".join(collected)
print("\n----- 누적된 완성본 -----")
print(full_text)

FastAPI는 **파이썬으로 빠르고 현대적인 웹 API를 쉽게 만들 수 있게 해주는 고성능 프레임워크**입니다.

----- 누적된 완성본 -----
FastAPI는 **파이썬으로 빠르고 현대적인 웹 API를 쉽게 만들 수 있게 해주는 고성능 프레임워크**입니다.


포인트

- `collected = []` 리스트에 조각을 모은 뒤 `"".join(...)`으로 합치는 게 문자열 `+=`보다 빠름
- 실시간 출력과 누적을 **동시에** 할 수 있음 — 받는 즉시 화면에 흘리면서 변수에도 쌓음
- 이 `full_text`를 그대로 DB에 저장하거나 로그로 남기면 됨

## 5. `stream_options`로 토큰 사용량 받기

스트리밍 응답은 기본적으로 `usage` 정보가 빠지거나, 환경에 따라 빈 chunk로만 도착합니다. `stream_options={"include_usage": True}` 옵션을 명시적으로 추가하면 **마지막 chunk**에 `usage`가 안정적으로 담겨 옵니다.

In [8]:
response = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": "Pydantic의 세줄 정도로 설명해줘."}],
    stream=True,
    stream_options={"include_usage": True},
)

collected = []
usage = None

for chunk in response:
    if chunk.choices:
        piece = chunk.choices[0].delta.content
        if piece:
            collected.append(piece)
            print(piece, end="", flush=True)
    if chunk.usage:
        usage = chunk.usage

print()
print("\n----- usage -----")
print(usage)

Pydantic는 **Python에서 데이터 검증과 설정 관리를 쉽게 해주는 라이브러리**입니다.  
주로 **타입 힌트 기반으로 입력 데이터를 검사하고 자동 변환**해, API나 백엔드 개발에서 많이 사용됩니다.  
예를 들어 JSON 요청 데이터를 **안전한 Python 객체로 바꿔주고**, 잘못된 값이 들어오면 **명확한 에러를 반환**해줍니다.

----- usage -----
CompletionUsage(completion_tokens=102, prompt_tokens=18, total_tokens=120, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


포인트

- `if chunk.choices:` — usage만 담긴 빈 chunk를 안전하게 건너뜀
- `if chunk.usage:` — usage 정보가 들어 있는 chunk에서만 보관
- 과금·로그·모니터링이 필요한 서비스에서는 거의 필수 옵션

# TODO

지금까지 본 패턴을 직접 손에 익히기 위한 과제입니다.

## TODO 1. chunk 누적 코드의 가드 채우기

아래 코드는 chunk를 누적하는 과정에서 두 가지 가드가 빠져 있습니다. 빈 chunk나 `None` 조각이 도착하면 에러가 발생하거나 잘못된 값이 누적됩니다. 두 곳을 채워 안전하게 동작하도록 만드세요.

- TODO 1-1: `chunk.choices`가 비어 있는 chunk는 건너뛰기
- TODO 1-2: `piece`가 유효한 값일 때만 `collected`에 추가 (`None`·빈 문자열은 제외)

> `continue` 또는 `if ...:` 조건을 활용하면 됩니다.

In [9]:
response = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": "REST API를 한 줄로 설명해줘."}],
    stream=True,
)

collected = []

for chunk in response:
    # TODO 1-1: choices가 비어 있는 chunk는 건너뛰기
    if not chunk.choices:
        continue
    piece = chunk.choices[0].delta.content
    # TODO 1-2: piece가 유효한 값일 때만 collected에 추가
    if piece:
        collected.append(piece)
        print(piece, end="", flush=True)
    
print()
full_text = "".join(collected)
print(full_text)

REST API는 웹에서 자원을 URL로 식별하고, HTTP 메서드(GET, POST, PUT, DELETE 등)로 상태를 주고받는 방식의 API입니다.
REST API는 웹에서 자원을 URL로 식별하고, HTTP 메서드(GET, POST, PUT, DELETE 등)로 상태를 주고받는 방식의 API입니다.


## TODO 2. 스트리밍 호출을 함수로 감싸기

`stream_text(message)` 함수를 만들어, 사용자 메시지를 받으면 다음을 반환하도록 하세요.

- `full_text`: 누적된 완성 답변
- `usage`: 마지막 chunk에서 받은 토큰 사용량 객체 (없으면 `None`)

조건

- 내부에서 `stream=True` + `stream_options={"include_usage": True}`로 호출
- chunk를 받는 즉시 화면에도 흘려보내기 (실시간 출력)
- `choices`가 비어 있는 chunk·`None` 조각 모두 안전하게 처리
- 함수 종료 시점에 `(full_text, usage)` 튜플 반환

In [14]:
def stream_text(message: str):

    # TODO 2: 위 조건을 만족하는 함수 본문을 작성하세요
    response = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": message}],
    stream=True,
    stream_options={"include_usage": True},
    )

    usage = None
    collected = []
    for chunk in response:
        if chunk.choices:
            piece = chunk.choices[0].delta.content
            if piece:
                collected.append(piece)
                print(piece, end="", flush=True)
        if chunk.usage:
            usage = chunk.usage

    full_text = "".join(collected)

    return full_text, usage

# 동작 확인
full, usage = stream_text("GraphQL을 다섯 줄로 설명해줘.")
print("\n----- 누적된 완성본 -----")
print(full)
print("\n----- usage -----")
print(usage)

GraphQL은 API를 위한 쿼리 언어이자 서버와 클라이언트 사이의 데이터 조회 방식입니다.  
클라이언트가 필요한 데이터만 정확히 요청할 수 있어 과다 요청과 부족 요청 문제를 줄여줍니다.  
하나의 엔드포인트에서 다양한 데이터를 구조적으로 가져오거나 수정할 수 있습니다.  
스키마를 기반으로 데이터 타입과 가능한 요청 형태를 명확하게 정의합니다.  
프론트엔드 개발 생산성을 높이고, 복잡한 데이터 관계를 다루기 쉽게 해줍니다.
----- 누적된 완성본 -----
GraphQL은 API를 위한 쿼리 언어이자 서버와 클라이언트 사이의 데이터 조회 방식입니다.  
클라이언트가 필요한 데이터만 정확히 요청할 수 있어 과다 요청과 부족 요청 문제를 줄여줍니다.  
하나의 엔드포인트에서 다양한 데이터를 구조적으로 가져오거나 수정할 수 있습니다.  
스키마를 기반으로 데이터 타입과 가능한 요청 형태를 명확하게 정의합니다.  
프론트엔드 개발 생산성을 높이고, 복잡한 데이터 관계를 다루기 쉽게 해줍니다.

----- usage -----
CompletionUsage(completion_tokens=127, prompt_tokens=18, total_tokens=145, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


## 실습 정리

- `stream=True` 옵션 하나로 응답 객체가 chunk iterator로 바뀐다는 걸 직접 확인
- chunk 구조(`delta.content`, `finish_reason`)와 가드 패턴(`if not chunk.choices`, `if piece`) 익힘
- 받은 chunk를 실시간 출력 + 변수에 누적하는 표준 패턴 작성
- `stream_options={"include_usage": True}`로 토큰 사용량을 마지막 chunk에서 회수